In [4]:
import json
import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
import joblib
import warnings

# Desativar warnings de convergência do ARIMA para manter o console limpo
warnings.filterwarnings("ignore")

# ==========================================
# 1. CARREGAMENTO E TRATAMENTO DOS DADOS (Alinhado com seu SCRIPT 1)
# ==========================================
print("Carregando o histórico do Firebase...")
with open('consumo-conciente.json', 'r') as f:
    dados_firebase = json.load(f)

if "leituras" in dados_firebase:
    dados_raiz = dados_firebase["leituras"]
else:
    dados_raiz = dados_firebase

linhas_achatadas = []
for data, horas in dados_raiz.items():
    if isinstance(horas, dict):
        for hora, atributos in horas.items():
            if isinstance(atributos, dict):
                registro = atributos.copy()

                # Sistema de tolerância/fallbacks idêntico ao SCRIPT 1
                if 'consumo' in registro and 'consumo_total_pzem' not in registro:
                    registro['consumo_total_pzem'] = registro['consumo']
                elif 'consumo_acumulado' in registro and 'consumo_total_pzem' not in registro:
                    registro['consumo_total_pzem'] = registro['consumo_acumulado']

                linhas_achatadas.append(registro)

df = pd.DataFrame(linhas_achatadas)

# Garantir a existência da coluna alvo
if 'consumo_total_pzem' not in df.columns:
    df['consumo_total_pzem'] = 0.0

# Filtrar a série temporal (ARIMA treina diretamente no dado real univariado, sem normalização)
serie_consumo = df['consumo_total_pzem'].dropna().astype(float).values

print(f"Total de amostras encontradas para treino: {len(serie_consumo)}")

# ==========================================
# 2. TREINAMENTO DO MODELO (Com toda a base)
# ==========================================
print("\nIniciando o treinamento do modelo estatístico ARIMA...")

# Configuração da ordem (p, d, q) para tendência linear estável:
# p=1 (Autoregressivo), d=1 (Diferenciação para estabilizar tendência), q=1 (Média Móvel)
ordem_arima = (1, 1, 1)
modelo = ARIMA(serie_consumo, order=ordem_arima)
modelo_ajustado = modelo.fit()

print("Treinamento concluído com sucesso!")
print(modelo_ajustado.summary())

# ==========================================
# 3. SALVAMENTO DO MODELO AJUSTADO
# ==========================================
nome_arquivo_modelo = 'modelo_arima_energia.pkl'
joblib.dump(modelo_ajustado, nome_arquivo_modelo)

print("\n" + "="*50)
print(f"Modelo ARIMA salvo com sucesso!")
print(f"Arquivo gerado: '{nome_arquivo_modelo}'")
print("="*50)

Carregando o histórico do Firebase...
Total de amostras encontradas para treino: 3777

Iniciando o treinamento do modelo estatístico ARIMA...
Treinamento concluído com sucesso!
                               SARIMAX Results                                
Dep. Variable:                      y   No. Observations:                 3777
Model:                 ARIMA(1, 1, 1)   Log Likelihood               19892.053
Date:                Fri, 10 Jul 2026   AIC                         -39778.106
Time:                        13:24:46   BIC                         -39759.397
Sample:                             0   HQIC                        -39771.455
                               - 3777                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.9180      0.003 

In [5]:
from google.colab import files

files.download('modelo_arima_energia.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>